# Reproduce the TF Arabidopsis-primary baseline

Onboarding Tier 0: retrain the existing TensorFlow SLEAP model `cyl_arabidopsis_7-11DAG_primary_6nodes` end-to-end, evaluate on the held-out test split, and record a documented **TF reference baseline** (config + metrics + W&B run link). The new PyTorch (`sleap-nn`) pipeline is graded against a fresh baseline later; these TF numbers are the reference.

This notebook consolidates the canonical helper-notebook workflow into one file. Each section maps to an existing helper notebook:

| Section | Mirrors |
|---------|---------|
| 1. Fetch labels | `helper_notebooks/make_dataset_registry.ipynb` (download instead of register) |
| 2. Make splits | `helper_notebooks/make_train_test_splits_first.ipynb` |
| 3. Modify config | `helper_notebooks/modify_init_configs_second.ipynb` |
| 4. Train | `helper_notebooks/sleap_train_with_wandb_third.ipynb` |
| 5. Evaluate | the evaluate notebooks / `train.evaluate_model_and_generate_visuals` |

Section 0 (GPU check) is the only addition, because of the RTX 5080 risk noted below.

**Run order:** top to bottom, from the `sleap` conda env, launched from this folder.

**RTX 5080 risk:** the 5080 is a Blackwell GPU (sm_120); SLEAP 1.4.1's TensorFlow predates it. If Section 0 reports no GPU, stop and message Elizabeth before training.

**Decisions to confirm with Elizabeth:**
- Config fidelity: the committed `configs/initial_config.base.json` is a proven primary-root bottom-up config (from the Medicago-primary experiment), not the exact Arabidopsis run. To reproduce a specific original model, drop its `training_config.json` in as `configs/reference_training_config.json` (Section 3 prefers it). The original experiment swept `max_stride`.
- Split fractions: 80/10/10, seed 42 (Setup cell).
- Pixel scale: metrics are in pixels; get px/mm before converting to mm.

## Setup: imports, W&B constants, and paths

Mirrors the constants block in `helper_notebooks/sleap_train_with_wandb_third.ipynb`. Edit the knobs here. Paths are relative to this folder, so launch Jupyter from `experiments/2026-07-06-repro-arabidopsis-primary/` (or set `EXP_DIR` manually).

In [ ]:
import os
import json
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd

# --- W&B coordinates (shared across the lab) ---
ENTITY_NAME = "eberrigan-salk-institute-for-biological-studies"
PROJECT_NAME = "sleap-roots"
LABELS_REGISTRY = "sleap-roots-labels"
MODELS_REGISTRY = "sleap-roots-models"

# Existing labels artifact we reproduce against (already in W&B).
LABELS_ARTIFACT = "cyl_arabidopsis_7-11DAG_primary_6nodes_labels"

# Unique name for THIS reproduction (keep distinct from Elizabeth's runs).
EXPERIMENT_NAME = "anirudh-repro-cyl-arabidopsis-primary-2026-07-06"

# Split reproducibility.
SEED = 42
FRACTION_TRAIN = 0.8      # 80% train; remaining 20% split 50/50 -> 10% val, 10% test
VAL_TEST_FRACTION = 0.5
VERSION = 0

# --- Paths (relative to this notebook's folder) ---
EXP_DIR = Path.cwd()
DATA_DIR = EXP_DIR / "data"
LABELS_DIR = DATA_DIR / "labels"
SPLIT_DIR = DATA_DIR / "splits" / "train_test_split.v000"
CSV_PATH = DATA_DIR / "splits" / "train_test_splits.csv"
BASE_CONFIG = EXP_DIR / "configs" / "initial_config.base.json"
REFERENCE_CONFIG = EXP_DIR / "configs" / "reference_training_config.json"

for d in (DATA_DIR, LABELS_DIR, SPLIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("EXP_DIR:", EXP_DIR)
print("Base config found:", BASE_CONFIG.exists())

## 0. GPU check (run first on the RTX 5080)

Not part of the original notebooks. The 5080 is a Blackwell GPU (sm_120) and SLEAP 1.4.1's TensorFlow predates it, so confirm TF can actually use the card before training. If this reports no GPU, stop and report the TF version to Elizabeth.

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__, "| built with CUDA:", tf.test.is_built_with_cuda())
gpus = tf.config.list_physical_devices("GPU")
print("Visible GPUs (", len(gpus), "):", gpus)

if not gpus:
    print()
    print("[GATE FAILED] No GPU visible to TensorFlow. This is the expected failure")
    print("mode for a Blackwell RTX 5080 on SLEAP 1.4.1's TF. Stop and report the TF")
    print("version above to Elizabeth before training.")
else:
    with tf.device("/GPU:0"):
        a = tf.random.normal((1024, 1024))
        _ = float(tf.reduce_sum(tf.matmul(a, a)).numpy())
    print()
    print("[GATE PASSED] GPU executed a test matmul. You can train.")

## 1. Fetch labels from W&B

Adapts `helper_notebooks/make_dataset_registry.ipynb`. The original notebook *registers* a `.slp` from the Salk network share (`//multilab-na...`). We can't reach that share, so we instead *download* the already-registered artifact (it was saved with embedded images).

In [ ]:
import wandb

run = wandb.init(project=PROJECT_NAME, entity=ENTITY_NAME, job_type="fetch_artifact")
try:
    artifact = run.use_artifact(f"{LABELS_ARTIFACT}:latest")
    download_dir = Path(artifact.download(root=str(LABELS_DIR)))
    print("Downloaded to:", download_dir)
finally:
    run.finish()

# Prefer a packaged .slp (embedded images) so splits can be saved with images.
candidates = sorted(LABELS_DIR.rglob("*.pkg.slp")) + sorted(LABELS_DIR.rglob("*.slp"))
assert candidates, f"No .slp found under {LABELS_DIR}"
LABELS_SLP = candidates[0]
print("Labels file:", LABELS_SLP)

## 2. Make train/val/test splits

Adapts `helper_notebooks/make_train_test_splits_first.ipynb`: same `sleap.nn.data.training.split_labels_train_val` calls and the same CSV schema (`path`, `version`, `labeled_frames`, `split_type`) that `train.main` consumes. Splits are saved with embedded images and seed-pinned for reproducibility.

In [ ]:
import sleap

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

base = sleap.load_file(LABELS_SLP.as_posix())
user_labels = base.with_user_labels_only() if hasattr(base, "with_user_labels_only") else base
print(f"Loaded {len(user_labels)} user-labeled frames")

# 80/10/10: hold out (1 - FRACTION_TRAIN), then split that 50/50 into val/test.
labels_train, _, labels_rem, _ = sleap.nn.data.training.split_labels_train_val(user_labels, 1 - FRACTION_TRAIN)
labels_val, _, labels_test, _ = sleap.nn.data.training.split_labels_train_val(labels_rem, VAL_TEST_FRACTION)

train_path = SPLIT_DIR / "train.pkg.slp"
val_path = SPLIT_DIR / "val.pkg.slp"
test_path = SPLIT_DIR / "test.pkg.slp"
labels_train.save(train_path.as_posix(), with_images=True)
labels_val.save(val_path.as_posix(), with_images=True)
labels_test.save(test_path.as_posix(), with_images=True)

rows = [
    {"path": train_path.as_posix(), "version": VERSION, "labeled_frames": len(labels_train), "split_type": "train"},
    {"path": val_path.as_posix(), "version": VERSION, "labeled_frames": len(labels_val), "split_type": "val"},
    {"path": test_path.as_posix(), "version": VERSION, "labeled_frames": len(labels_test), "split_type": "test"},
]
pd.DataFrame(rows).to_csv(CSV_PATH, index=False)
print(f"train {len(labels_train)} | val {len(labels_val)} | test {len(labels_test)}")
print("wrote", CSV_PATH)

## 3. Modify the init config

Adapts `helper_notebooks/modify_init_configs_second.ipynb`: loads a base config, fills in the train/val/test label paths and `runs_folder`, and writes `initial_config_modified_v000.json` into the split dir (where `train.main` looks for it). If `configs/reference_training_config.json` exists (the original run's exact config), it is preferred over the default base.

In [ ]:
src = REFERENCE_CONFIG if REFERENCE_CONFIG.exists() else BASE_CONFIG
print("Using base config:", src.name)

cfg = copy.deepcopy(json.loads(Path(src).read_text()))
cfg["data"]["labels"]["training_labels"] = (SPLIT_DIR / "train.pkg.slp").as_posix()
cfg["data"]["labels"]["validation_labels"] = (SPLIT_DIR / "val.pkg.slp").as_posix()
cfg["data"]["labels"]["test_labels"] = (SPLIT_DIR / "test.pkg.slp").as_posix()
cfg["data"]["labels"]["training_inds"] = None
cfg["data"]["labels"]["validation_inds"] = None
cfg["data"]["labels"]["test_inds"] = None
cfg["outputs"]["runs_folder"] = (SPLIT_DIR / "models").as_posix()
cfg["outputs"]["run_name"] = f"{EXPERIMENT_NAME}-v00{VERSION}"

config_path = SPLIT_DIR / f"initial_config_modified_v00{VERSION}.json"
config_path.write_text(json.dumps(cfg, indent=2))
print("wrote", config_path)
print("max_stride:", cfg["model"]["backbone"]["unet"]["max_stride"],
      "| input_scaling:", cfg["data"]["preprocessing"]["input_scaling"],
      "| epochs:", cfg["optimization"]["epochs"])

## 4. Train

Mirrors `helper_notebooks/sleap_train_with_wandb_third.ipynb`: set the W&B config via `srt.config.update_config(...)`, then call `sleap_roots_training.train.main(...)`, which runs `sleap-train` on each CSV version, then auto-evaluates on the test split and logs a W&B model artifact.

Set `SMOKE_TEST = True` for a fast 2-epoch end-to-end check first, then rerun with `False` for the real run. Metrics are in pixels (`px_per_mm=None`). `link_to_registry=False` avoids the package's deprecated registry namespace; link separately if desired.

In [ ]:
import sleap_roots_training as srt
from sleap_roots_training.train import main as train_main

SMOKE_TEST = True  # set False for the full run

TAGS = ["cyl", "arabidopsis", "7-11DAG", "primary", "6nodes", "tf-baseline", "repro"]

if SMOKE_TEST:
    _p = SPLIT_DIR / f"initial_config_modified_v00{VERSION}.json"
    _c = json.loads(_p.read_text())
    _c["optimization"]["epochs"] = 2
    _c["optimization"]["min_batches_per_epoch"] = 10
    _c["optimization"]["min_val_batches_per_epoch"] = 2
    _p.write_text(json.dumps(_c, indent=2))
    TAGS = TAGS + ["smoke-test"]
    print("[smoke-test] config patched to 2 epochs")

srt.config.reset_config()
srt.config.update_config(
    entity_name=ENTITY_NAME,
    project_name=PROJECT_NAME,
    experiment_name=EXPERIMENT_NAME,
    registry=MODELS_REGISTRY,
    collection_name=EXPERIMENT_NAME,
    job_type="train",
)

train_main(
    csv_path=str(CSV_PATH),
    tags=TAGS,
    model_tags=TAGS,
    sleap_train_command="sleap-train {}",
    use_existing_model=False,
    use_sweep=False,
    link_to_registry=False,
    registry_name=None,
)

## 5. Read the held-out test metrics

`sleap-train` writes `metrics.{train,val,test}.npz` in the model dir (the same metrics `train.evaluate_model_and_generate_visuals` reads). These test-set numbers are the TF reference baseline: `dist.*` are localization error in pixels; `oks_voc.*` and `vis.*` are detection quality.

In [ ]:
WANTED = ["dist.p50", "dist.p90", "dist.p95", "dist.avg",
          "oks_voc.mAP", "oks_voc.mAR", "vis.precision", "vis.recall"]

def read_metrics(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    if "metrics" not in data.files:
        return {}
    m = data["metrics"].item()
    out = {}
    for k in WANTED:
        if k in m:
            try:
                out[k] = float(m[k])
            except (TypeError, ValueError):
                v = np.asarray(m[k])
                if v.dtype.kind in ("f", "i") and v.size:
                    out[k] = float(np.nanmean(v))
    return out

for model_dir in sorted((SPLIT_DIR / "models").glob("*")):
    print(f"=== {model_dir.name} ===")
    for split in ("train", "val", "test"):
        npz = list(model_dir.glob(f"metrics.{split}.npz"))
        if not npz:
            print(f"  {split:5s} (none)")
            continue
        m = read_metrics(npz[0])
        print(f"  {split:5s} " + "  ".join(f"{k}={m[k]:.4f}" for k in WANTED if k in m))

## 6. Record the baseline

Copy the test-set numbers above, plus the W&B run URL (printed during training / visible in the run page), into `TF_REFERENCE_BASELINE.md` in this folder. That file is the deliverable: config + metrics + W&B link + a short note on how close this is to the original run.